# SCRUM-102 · Entrenamiento del clasificador (Lina's Pet Salón)

CNN EfficientNet-B0 por transfer learning que predice **tamaño**, **grupo de manto** (A–D del tarifario v2) y **estado del pelo**.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.

En Google Drive debe existir `MyDrive/MIA/dataset/` con:
- `imagenes/` → las 312 fotos (parte 1 + parte 2 juntas)
- `manifiesto_v2.csv` → generado con `preetiquetar.py manifiesto` (solo filas revisadas por el equipo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET = '/content/drive/MyDrive/MIA/dataset'
SALIDA = '/content/drive/MyDrive/MIA/modelos'
!ls $DATASET | head

In [ ]:
# Código del proyecto (el repo puede ser privado: usar un token de GitHub de solo lectura)
!git clone -b prototipo-s4 https://github.com/nicopolis4992/Proyecto-MIA.git mia
!pip install -q onnx onnxruntime scikit-learn

In [ ]:
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU - cambiar el entorno')

In [ ]:
!cd mia && python entrenamiento/clasificador_imagen/entrenar_colab.py \n    --manifiesto $DATASET/manifiesto_v2.csv \n    --raiz-imagenes $DATASET \n    --salida $SALIDA \n    --epocas-cabezas 4 --epocas-ajuste 12

## Resultado

En `MyDrive/MIA/modelos/` quedan `clasificador_v1.onnx` y `clasificador_v1.json` (clases, temperatura de calibración y métricas F1).

Para usarlo en el bot: copiarlos a `app/vision/modelos/` y agregar `onnxruntime` a `requirements.txt`. Con `VISION_BACKEND=auto`, el sistema usa la CNN en lugar de Gemini.

In [ ]:
import json
print(json.dumps(json.load(open(f'{SALIDA}/clasificador_v1.json'))['metricas'], indent=2))